# Maximum Likelihood Estimation

## Learning Objectives
1. Derive and implement MLE analytically for Gaussian parameters and verify numerically
2. Visualize the log-likelihood surface and understand curvature (Fisher information)
3. Apply MLE to logistic regression via gradient ascent and compare to sklearn
4. Demonstrate the cross-entropy = negative log-likelihood equivalence numerically

In [ ]:
import numpy as np
import scipy.stats as stats
import scipy.optimize as opt
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, Ridge

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Imports OK')
print('NumPy:', np.__version__)

## Level 1: MLE for Gaussian Parameters from Scratch

For X_i ~ N(mu, sigma^2), the MLE solutions are:
- mu_hat = (1/n) sum x_i  (sample mean)
- sigma_hat^2 = (1/n) sum (x_i - mu_hat)^2  (biased; unbiased uses n-1)

We verify by plotting the log-likelihood surface and finding its maximum.

In [ ]:
# --- Generate data ---
true_mu = 4.0
true_sigma = 1.5
n = 50
data = np.random.normal(true_mu, true_sigma, n)
print(f'True: mu={true_mu}, sigma={true_sigma}')
print(f'Data: n={n}, sample mean={data.mean():.4f}, sample std={data.std():.4f}')

# --- Analytical MLE ---
mu_mle    = data.mean()                         # = (1/n) sum x_i
sigma_mle = np.sqrt(np.mean((data - mu_mle)**2)) # = sqrt( (1/n) sum (x_i - mu_hat)^2 )
sigma_unbiased = data.std(ddof=1)               # unbiased: uses n-1

print(f'MLE mu_hat     = {mu_mle:.4f}  (true: {true_mu})')
print(f'MLE sigma_hat  = {sigma_mle:.4f}  (true: {true_sigma})')
print(f'Unbiased sigma = {sigma_unbiased:.4f}  (uses n-1 denominator)')
print(f'Bias in sigma_mle: {sigma_mle - true_sigma:.4f}')

# --- Log-likelihood function ---
def log_likelihood_gaussian(mu, sigma, data):
    # sum log N(x_i; mu, sigma^2)
    n_ = len(data)
    return (-n_/2 * np.log(2 * np.pi)
            - n_ * np.log(sigma)
            - np.sum((data - mu)**2) / (2 * sigma**2))

ll_at_mle  = log_likelihood_gaussian(mu_mle, sigma_mle, data)
ll_at_true = log_likelihood_gaussian(true_mu, true_sigma, data)
print(f'Log-likelihood at MLE:  {ll_at_mle:.4f}')
print(f'Log-likelihood at true: {ll_at_true:.4f}')
print(f'MLE >= true? {ll_at_mle >= ll_at_true} (MLE maximizes by definition)')

# --- 2D log-likelihood surface ---
mu_grid    = np.linspace(2.5, 5.5, 100)
sigma_grid = np.linspace(0.5, 3.0, 100)
MU, SG     = np.meshgrid(mu_grid, sigma_grid)
LL         = np.array([[log_likelihood_gaussian(m, s, data) for m in mu_grid]
                        for s in sigma_grid])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contour plot
cs = axes[0].contourf(MU, SG, LL, levels=30, cmap='viridis')
axes[0].scatter(mu_mle, sigma_mle, color='red', s=100, zorder=5, label=f'MLE ({mu_mle:.2f}, {sigma_mle:.2f})')
axes[0].scatter(true_mu, true_sigma, color='white', s=100, marker='*', zorder=6, label=f'True ({true_mu}, {true_sigma})')
axes[0].set_xlabel('mu')
axes[0].set_ylabel('sigma')
axes[0].set_title('Log-Likelihood Surface N(mu, sigma^2)', fontweight='bold')
axes[0].legend(fontsize=9)
plt.colorbar(cs, ax=axes[0])

# Profile log-likelihood for mu (fixing sigma at MLE)
ll_profile_mu = [log_likelihood_gaussian(m, sigma_mle, data) for m in mu_grid]
axes[1].plot(mu_grid, ll_profile_mu, 'b-', lw=2, label='Profile log-likelihood')
axes[1].axvline(mu_mle, color='red', ls='--', lw=2, label=f'MLE mu={mu_mle:.3f}')
axes[1].axvline(true_mu, color='black', ls=':', lw=2, label=f'True mu={true_mu}')
# Fisher information: -d^2 ell / d mu^2 = n / sigma^2
fisher_mu = n / sigma_mle**2
axes[1].set_xlabel('mu')
axes[1].set_ylabel('Log-likelihood')
axes[1].set_title(f'Profile Log-Likelihood for mu\nFisher I(mu)={fisher_mu:.2f}, SE={1/np.sqrt(fisher_mu):.4f}', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_04_gaussian_mle.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_04_gaussian_mle.png')

# --- Asymptotic confidence interval from Fisher information ---
# SE(mu_hat) = sigma / sqrt(n) = 1 / sqrt(I(mu))
se_mu = sigma_mle / np.sqrt(n)
ci_lo = mu_mle - 1.96 * se_mu
ci_hi = mu_mle + 1.96 * se_mu
print(f'95% CI for mu: [{ci_lo:.3f}, {ci_hi:.3f}]  (true mu={true_mu} inside: {ci_lo <= true_mu <= ci_hi})')

## Level 2: MLE for Logistic Regression via Gradient Ascent

Logistic regression uses MLE with Bernoulli likelihood:
  log L(w) = sum_i [ y_i log sigma(w^T x_i) + (1-y_i) log(1-sigma(w^T x_i)) ]

This is CONCAVE in w, so gradient ascent always finds the global maximum.
We implement from scratch and verify against sklearn.

In [ ]:
np.random.seed(42)

# --- Generate binary classification data ---
n_samples = 400
n_features = 3
w_true = np.array([2.0, -1.5, 1.0])
X = np.random.randn(n_samples, n_features)
logits = X @ w_true
probs = 1 / (1 + np.exp(-logits))
y = (np.random.rand(n_samples) < probs).astype(float)
print(f'Binary data: n={n_samples}, features={n_features}')
print(f'Class balance: {y.mean():.3f} positive')

# --- Log-likelihood for logistic regression ---
def sigmoid(z):
    # Numerically stable sigmoid
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))

def log_likelihood_logistic(w, X_, y_):
    # sum y_i log p_i + (1-y_i) log(1-p_i)
    p = sigmoid(X_ @ w)
    # Clip for numerical safety
    p = np.clip(p, 1e-10, 1 - 1e-10)
    return float(np.sum(y_ * np.log(p) + (1 - y_) * np.log(1 - p)))

def gradient_log_likelihood(w, X_, y_):
    # grad = X^T (y - p)  -- elegant closed form
    p = sigmoid(X_ @ w)
    return X_.T @ (y_ - p)

# --- Manual gradient ascent ---
lr = 0.1
n_iters = 300
w_ga = np.zeros(n_features)   # start from zero
ll_history = []

for i in range(n_iters):
    grad = gradient_log_likelihood(w_ga, X, y)
    w_ga = w_ga + lr * grad
    if i % 50 == 0:
        ll = log_likelihood_logistic(w_ga, X, y)
        ll_history.append(ll)

final_ll = log_likelihood_logistic(w_ga, X, y)
print(f'Gradient ascent (lr={lr}, {n_iters} iters): w={w_ga.round(3)}')
print(f'Final log-likelihood: {final_ll:.3f}')

# --- scipy.optimize.minimize (L-BFGS-B) ---
def neg_ll(w):
    return -log_likelihood_logistic(w, X, y)

def neg_grad(w):
    return -gradient_log_likelihood(w, X, y)

result = opt.minimize(neg_ll, x0=np.zeros(n_features), jac=neg_grad, method='L-BFGS-B')
w_lbfgs = result.x
print(f'L-BFGS-B:             w={w_lbfgs.round(3)}')

# --- sklearn baseline ---
lr_sk = LogisticRegression(C=1e6, solver='lbfgs', random_state=42)  # C=1e6 = minimal regularization
lr_sk.fit(X, y)
w_sk = lr_sk.coef_[0]
print(f'sklearn (C=1e6):       w={w_sk.round(3)}')
print(f'True weights:          w={w_true}')

# --- Fisher information = Hessian of log-likelihood (for confidence intervals) ---
# For logistic: I(w) = X^T diag(p*(1-p)) X
p_opt = sigmoid(X @ w_lbfgs)
W_diag = p_opt * (1 - p_opt)         # elementwise weights
fisher_info = X.T @ (W_diag[:, None] * X)  # X^T W X
cov_hat  = np.linalg.inv(fisher_info) # asymptotic covariance
std_errs = np.sqrt(np.diag(cov_hat))

print('\nMLE estimates with asymptotic 95% CI:')
for j in range(n_features):
    ci_lo = w_lbfgs[j] - 1.96 * std_errs[j]
    ci_hi = w_lbfgs[j] + 1.96 * std_errs[j]
    in_ci = ci_lo <= w_true[j] <= ci_hi
    print(f'  w[{j}]: {w_lbfgs[j]:.3f} +/- {std_errs[j]:.3f}  [{ci_lo:.3f}, {ci_hi:.3f}]  true={w_true[j]}  covered={in_ci}')

# Plot log-likelihood convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ll_full = [log_likelihood_logistic(np.zeros(n_features) + i/n_iters * w_ga, X, y)
           for i in range(n_iters)]
axes[0].plot(range(n_iters), ll_full, 'b-', lw=1.5, label='Gradient ascent')
axes[0].axhline(final_ll, color='red', ls='--', label=f'Converged LL={final_ll:.1f}')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Log-likelihood')
axes[0].set_title('Logistic Regression MLE Convergence', fontweight='bold')
axes[0].legend()

axes[1].scatter(w_true, w_lbfgs, s=100, color='blue')
diag_range = np.linspace(-2.5, 2.5, 100)
axes[1].plot(diag_range, diag_range, 'r--', label='Perfect recovery')
axes[1].errorbar(w_true, w_lbfgs, yerr=1.96*std_errs, fmt='none', color='blue', capsize=5)
for j in range(n_features):
    axes[1].annotate(f'w[{j}]', (w_true[j], w_lbfgs[j]), textcoords='offset points', xytext=(8,4))
axes[1].set_xlabel('True weights'); axes[1].set_ylabel('MLE estimates')
axes[1].set_title('Weight Recovery with 95% CI error bars', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_04_logistic_mle.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_04_logistic_mle.png')

## Real-World Example 1: Cross-Entropy = Negative Log-Likelihood

Cross-entropy H(y, p) = -sum_k y_k log p_k is the negative log-likelihood
of the categorical (softmax) distribution.

Minimizing cross-entropy in training is exactly MLE.
We verify this equivalence numerically on a multi-class classification problem.

In [ ]:
np.random.seed(42)

# --- Synthetic 3-class data ---
n = 300
n_classes = 3

# Simulate softmax outputs (pseudo-predictions)
logits = np.random.randn(n, n_classes)

def softmax(z):
    # Numerically stable: subtract max before exp
    z_shift = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

probs = softmax(logits)  # shape (n, n_classes)

# True labels (one-hot)
true_classes = np.argmax(probs, axis=1)  # assign most likely class as ground truth
y_onehot = np.zeros((n, n_classes))
y_onehot[np.arange(n), true_classes] = 1

# --- Cross-entropy loss (average) ---
def cross_entropy(y_oh, p):
    # H = -mean sum_k y_k log p_k
    p_clip = np.clip(p, 1e-10, 1.0)
    return -np.mean(np.sum(y_oh * np.log(p_clip), axis=1))

# --- Negative log-likelihood ---
def neg_log_likelihood(y_oh, p):
    # NLL = -mean log P(class = y_i | x_i)
    # = -mean log p[i, true_class_i]
    p_clip = np.clip(p, 1e-10, 1.0)
    true_class_probs = p_clip[np.arange(n), true_classes]
    return -np.mean(np.log(true_class_probs))

ce  = cross_entropy(y_onehot, probs)
nll = neg_log_likelihood(y_onehot, probs)

print(f'Cross-entropy loss:    {ce:.6f}')
print(f'Negative log-likelihood: {nll:.6f}')
print(f'Difference: {abs(ce - nll):.2e}  (should be ~0)')
print(f'They are identical: {np.isclose(ce, nll)}')

# --- Demonstrate gradient of cross-entropy = gradient of NLL ---
# For softmax + cross-entropy, the gradient wrt logits is p - y (clean closed form)
# This is the MLE gradient for categorical distribution
grad_logits = (probs - y_onehot) / n  # same as d(CE)/d(logits)
print(f'\nGradient shape: {grad_logits.shape}')
print(f'Mean gradient norm: {np.linalg.norm(grad_logits, axis=1).mean():.4f}')

# --- Show how CE changes as predictions improve ---
print('\nCross-entropy vs prediction quality:')
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
for temp in temperatures:
    probs_temp = softmax(logits / temp)
    ce_temp = cross_entropy(y_onehot, probs_temp)
    max_probs = probs_temp.max(axis=1).mean()
    print(f'  Temperature={temp}: CE={ce_temp:.3f}, avg max prob={max_probs:.3f}')

print('\nKey insight: lower temperature = more confident predictions = lower CE')

# --- MLE loss landscape: vary a single logit weight ---
w_range = np.linspace(-4, 4, 200)
# Simple: binary case, single weight
X_simple = np.random.randn(200, 1)
y_simple = (X_simple[:, 0] > 0).astype(float)
ll_curve = []
for w in w_range:
    p = sigmoid(X_simple[:, 0] * w)
    p = np.clip(p, 1e-10, 1 - 1e-10)
    ll = np.sum(y_simple * np.log(p) + (1 - y_simple) * np.log(1 - p))
    ll_curve.append(ll)

w_opt_idx = np.argmax(ll_curve)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(w_range, ll_curve, 'b-', lw=2)
axes[0].axvline(w_range[w_opt_idx], color='red', ls='--', lw=2, label=f'MLE w={w_range[w_opt_idx]:.2f}')
axes[0].set_xlabel('Weight w'); axes[0].set_ylabel('Log-likelihood')
axes[0].set_title('Binary Log-Likelihood is Concave in w\n(single global maximum)', fontweight='bold')
axes[0].legend()

# CE vs NLL scatter (should be identity line)
ce_values  = [cross_entropy(y_onehot, softmax(logits * t)) for t in [0.3, 0.5, 1.0, 1.5, 2.0]]
nll_values = [neg_log_likelihood(y_onehot, softmax(logits * t)) for t in [0.3, 0.5, 1.0, 1.5, 2.0]]
axes[1].scatter(ce_values, nll_values, s=100, color='blue')
axes[1].plot([min(ce_values), max(ce_values)], [min(ce_values), max(ce_values)],
             'r--', label='CE = NLL')
axes[1].set_xlabel('Cross-Entropy'); axes[1].set_ylabel('Neg. Log-Likelihood')
axes[1].set_title('Cross-Entropy = Negative Log-Likelihood', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_04_ce_vs_nll.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_04_ce_vs_nll.png')

## Real-World Example 2: MLE for Poisson Rate with Confidence Interval

For Poisson data, MLE has a simple analytical solution: lambda_hat = x_bar.
The Fisher information for Poisson is I(lambda) = n/lambda,
so SE(lambda_hat) = sqrt(lambda_hat / n).

In [ ]:
np.random.seed(42)

# --- Simulate web server request rate (Poisson process) ---
true_lambda = 47.3  # requests per second
n_obs = 60          # 60 seconds of data
data_poisson = np.random.poisson(true_lambda, n_obs)

# --- Analytical MLE for Poisson ---
# ell(lambda) = sum(x_i) * log(lambda) - n * lambda - sum(log(x_i!))
# d ell / d lambda = sum(x_i)/lambda - n = 0  =>  lambda_hat = x_bar
lambda_hat = data_poisson.mean()
print(f'True lambda:  {true_lambda}')
print(f'MLE lambda:   {lambda_hat:.3f}  (sample mean, as expected)')

# --- Fisher information for Poisson ---
# I(lambda) = n / lambda  (derived from E[-d^2 ell / d lambda^2])
# SE(lambda_hat) = sqrt(lambda_hat / n)
fisher_poisson = n_obs / lambda_hat
se_lambda      = np.sqrt(lambda_hat / n_obs)
ci_95_lo = lambda_hat - 1.96 * se_lambda
ci_95_hi = lambda_hat + 1.96 * se_lambda
print(f'Fisher I(lambda): {fisher_poisson:.3f}')
print(f'SE(lambda_hat):   {se_lambda:.4f}')
print(f'95% CI:  [{ci_95_lo:.3f}, {ci_95_hi:.3f}]'
      f'  Contains true? {ci_95_lo <= true_lambda <= ci_95_hi}')

# --- Coverage probability: does 95% CI actually cover 95% of the time? ---
n_simulations = 5000
covered = []
for _ in range(n_simulations):
    sim_data   = np.random.poisson(true_lambda, n_obs)
    lam_est    = sim_data.mean()
    se_est     = np.sqrt(lam_est / n_obs)
    lo = lam_est - 1.96 * se_est
    hi = lam_est + 1.96 * se_est
    covered.append(lo <= true_lambda <= hi)

coverage = np.mean(covered)
print(f'\nEmpirical CI coverage: {coverage:.3f} (target: 0.95)')
print('Coverage close to 0.95 validates asymptotic approximation')

# --- Profile log-likelihood for Poisson ---
def poisson_log_lik(lam, data):
    # sum x_i * log(lam) - n * lam   (dropping constant log(x_i!) term)
    if lam <= 0:
        return -np.inf
    return float(data.sum() * np.log(lam) - len(data) * lam)

lam_range = np.linspace(30, 70, 400)
ll_vals   = [poisson_log_lik(lam, data_poisson) for lam in lam_range]

# --- Compare different sample sizes ---
print('\nMLE precision vs sample size:')
for n_ in [10, 30, 60, 200, 1000]:
    d_ = np.random.poisson(true_lambda, n_)
    lh = d_.mean()
    se_ = np.sqrt(lh / n_)
    print(f'  n={n_:<6}: lambda_hat={lh:.2f}  SE={se_:.3f}  95% CI width={3.92*se_:.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(lam_range, ll_vals, 'b-', lw=2)
axes[0].axvline(lambda_hat, color='red', ls='--', lw=2, label=f'MLE={lambda_hat:.2f}')
axes[0].axvline(true_lambda, color='black', ls=':', lw=2, label=f'True={true_lambda}')
axes[0].axvspan(ci_95_lo, ci_95_hi, alpha=0.15, color='red', label='95% CI')
axes[0].set_xlabel('lambda'); axes[0].set_ylabel('Log-likelihood')
axes[0].set_title('Poisson Log-Likelihood Profile', fontweight='bold')
axes[0].legend(fontsize=9)

# CI width vs sample size
n_sizes = np.arange(5, 501)
ci_widths = 2 * 1.96 * np.sqrt(lambda_hat / n_sizes)
axes[1].plot(n_sizes, ci_widths, 'b-', lw=2)
axes[1].axvline(n_obs, color='red', ls='--', label=f'Our n={n_obs}')
axes[1].set_xlabel('Sample size n'); axes[1].set_ylabel('95% CI width')
axes[1].set_title('Precision Improves as sqrt(n)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stats_04_poisson_mle.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_04_poisson_mle.png')

## Real-World Example 3: MLE vs MAP (L2 Regularization)

Adding a Gaussian prior on weights converts MLE to MAP:
  MAP = argmax [ log-likelihood + log-prior ]
      = argmax [ log-likelihood - lambda * ||w||^2 ]

This is Ridge regression. We compare MLE vs MAP on an overfit problem
and show AIC/BIC model selection.

In [ ]:
np.random.seed(42)

# --- Overfit scenario: n=40 samples, 20 features ---
n_train = 40
n_test  = 1000
d = 20

# True: only first 3 features matter
w_true = np.zeros(d)
w_true[:3] = [3.0, -2.0, 1.5]
noise_std = 0.5

X_tr = np.random.randn(n_train, d)
X_te = np.random.randn(n_test, d)
y_tr = X_tr @ w_true + np.random.randn(n_train) * noise_std
y_te = X_te @ w_true + np.random.randn(n_test) * noise_std

# --- MLE (OLS) ---
XtX = X_tr.T @ X_tr
Xty = X_tr.T @ y_tr
try:
    w_ols = np.linalg.solve(XtX, Xty)
except np.linalg.LinAlgError:
    w_ols = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]

rmse_train_ols = np.sqrt(np.mean((X_tr @ w_ols - y_tr)**2))
rmse_test_ols  = np.sqrt(np.mean((X_te @ w_ols - y_te)**2))

# --- MAP (Ridge) ---
lambdas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results = {}
for lam in lambdas:
    w_r = np.linalg.solve(XtX + lam * np.eye(d), Xty)
    tr  = np.sqrt(np.mean((X_tr @ w_r - y_tr)**2))
    te  = np.sqrt(np.mean((X_te @ w_r - y_te)**2))
    # Log-likelihood at Ridge solution
    resid = y_tr - X_tr @ w_r
    sigma_hat = resid.std()
    if sigma_hat < 1e-10: sigma_hat = 1e-10
    ll = stats.norm.logpdf(resid, scale=sigma_hat).sum()
    k  = np.sum(np.abs(w_r) > 0.001)  # effective params (approx)
    aic = 2 * d - 2 * ll
    bic = d * np.log(n_train) - 2 * ll
    ridge_results[lam] = {'w': w_r, 'train': tr, 'test': te, 'aic': aic, 'bic': bic}

print(f'MLE (OLS): train RMSE={rmse_train_ols:.3f}  test RMSE={rmse_test_ols:.3f}  (overfit!)')
print(f'\nMAP (Ridge) results:')
print(f'  {'lambda':<10} {'Train RMSE':>12} {'Test RMSE':>12} {'AIC':>10} {'BIC':>10}')
print('  ' + '-' * 58)
for lam, r in sorted(ridge_results.items()):
    print(f'  {lam:<10} {r["train"]:>12.3f} {r["test"]:>12.3f} {r["aic"]:>10.1f} {r["bic"]:>10.1f}')

best_lam = min(ridge_results, key=lambda l: ridge_results[l]['test'])
print(f'\nBest lambda by test RMSE: {best_lam}')
print(f'MAP reduces test RMSE by {(rmse_test_ols - ridge_results[best_lam]["test"])/rmse_test_ols*100:.1f}% vs MLE')

# --- Comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Weight comparison: MLE vs best MAP
w_best = ridge_results[best_lam]['w']
feat_idx = np.arange(d)
axes[0].bar(feat_idx - 0.2, w_true, width=0.4, color='black', alpha=0.7, label='True w')
axes[0].bar(feat_idx + 0.2, w_ols,  width=0.4, color='red',   alpha=0.5, label='MLE w')
axes[0].bar(feat_idx,       w_best, width=0.2, color='blue',  alpha=0.7, label=f'MAP w (lam={best_lam})')
axes[0].set_xlabel('Feature'); axes[0].set_ylabel('Weight')
axes[0].set_title('Weight Comparison: True / MLE / MAP', fontweight='bold')
axes[0].legend(fontsize=8)

# Test RMSE vs lambda
lam_list = list(ridge_results.keys())
test_rmses = [ridge_results[l]['test'] for l in lam_list]
train_rmses = [ridge_results[l]['train'] for l in lam_list]
axes[1].semilogx(lam_list, train_rmses, 'g-o', lw=2, label='Train RMSE')
axes[1].semilogx(lam_list, test_rmses, 'b-o', lw=2, label='Test RMSE')
axes[1].axhline(rmse_test_ols, color='red', ls='--', label=f'MLE test RMSE={rmse_test_ols:.2f}')
axes[1].axvline(best_lam, color='gray', ls=':', label=f'Best lambda={best_lam}')
axes[1].set_xlabel('Lambda (regularization strength)')
axes[1].set_ylabel('RMSE')
axes[1].set_title('MLE overfits; MAP (Ridge) generalizes', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('stats_04_map_vs_mle.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_04_map_vs_mle.png')

print('\nKEY TAKEAWAYS for MLE:')
print('1. MLE = argmax log-likelihood (log for numerical stability)')
print('2. Analytical solutions exist: Gaussian, Binomial, Poisson')
print('3. Cross-entropy loss = negative log-likelihood for categorical')
print('4. Fisher info = expected curvature = inverse asymptotic variance')
print('5. MLE is consistent + asymptotically efficient as n -> infinity')
print('6. MLE overfits when n << d; add prior (MAP/Ridge) to regularize')
print('7. Use AIC/BIC to compare models: penalize parameter count')

print('\nEXERCISES:')
print('1. Derive the MLE for Exponential(lambda) and verify with scipy')
print('2. Show that MSE loss = MLE under Gaussian noise assumption')
print('3. Try Laplace prior (L1) instead of Gaussian (L2) and compare sparsity')